In [1]:
import os

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
from typing import Literal
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
memory = InMemorySaver()

In [4]:
from langchain_core.messages import SystemMessage
llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)

In [5]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


In [6]:
def language_translator(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    system = SystemMessage(content="You are a helpful and friendly assistant that translates English language to French")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}

In [7]:
builder = StateGraph(State)

In [8]:
       
builder.add_node("chat", language_translator)

In [9]:
builder.add_edge(START, "chat")

In [10]:
builder.add_edge("chat", END)

In [11]:
router = builder.compile(checkpointer=memory)

In [12]:
config = {"configurable": {"thread_id": "user_123"}}

In [13]:
result1 = router.invoke(
    {"messages": [HumanMessage(content="My name is Alice and I love astronomy")]},
    config=config            # ← thread_id = "user_123"
)

In [14]:
print(result1["messages"][-1].content)

Je m'appelle Alice et j'aime l'astronomie.


In [16]:
result2 = router.invoke(
    {"messages": [HumanMessage(content="I school in the nertherlands and I love astronomy")]},
    config=config            # ← thread_id = "user_123"
)

In [17]:
print(result2["messages"][-1].content)

Je suis à l'école aux Pays-Bas et j'aime l'astronomie.


In [18]:
result3 = router.invoke(
    {"messages": [HumanMessage(content="What is my name?")]},
    config=config            # ← thread_id = "user_123"
)

In [19]:
print(result3["messages"][-1].content)

Your name is Alice.
